# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` â€” folds, track preparation, metrics, the loop itself
- `src/models/` â€” model functions and `MODEL_REGISTRY`
- `src/features/quantum/` â€” quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: C:\dev\projects\capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "Jack"  # your name, recorded on every result row (default: OS user)
RESUME        = True  # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce",
                  "Q_heisenberg_pauli", "Q_heisenberg_single_bit", "Q_zz_pauli", "Q_zz_pce", "Q_zz_single_bit",
                  "MCR_su_z"]  # multi-circuit reservoir: notebooks/quantum_reservoir_prep.ipynb
ACTIVE_TRACKS = ["A", "B", *QUANTUM_TRACKS]                    # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["random_walk", "AR6",
                 "decision_tree", "adaboost", "random_forest", "xgboost",
                 "linear_svm", "sig_kernel", "linear_readout",
                 "rnn", "gru", "lstm"]  # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['A', 'B', 'Q_su_pce', 'Q_su_pauli', 'Q_su_bitstring', 'Q_su_single_bit', 'Q_heisenberg_pce', 'Q_heisenberg_pauli', 'Q_heisenberg_single_bit', 'Q_zz_pauli', 'Q_zz_pce', 'Q_zz_single_bit', 'MCR_su_z']
Models:   ['random_walk', 'AR6', 'decision_tree', 'adaboost', 'random_forest', 'xgboost', 'linear_svm', 'sig_kernel', 'linear_readout', 'rnn', 'gru', 'lstm']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track A        target INDPRO     loaded   shape=(767, 123)
  Track A        target PAYEMS     loaded   shape=(767, 123)
  Track A        target CPIAUCSL   loaded   shape=(767, 123)
  Track A        target S&P 500    loaded   shape=(767, 123)
  Track B        target INDPRO     loaded   shape=(767, 10)
  Track B        target PAYEMS     loaded   shape=(767, 10)
  Track B        target CPIAUCSL   loaded   shape=(767, 12)
  Track B        target S&P 500    loaded   shape=(767, 12)
  Track Q_su_pce target INDPRO     loaded   shape=(767, 9)
  Track Q_su_pce target PAYEMS     loaded   shape=(767, 9)
  Track Q_su_pce target CPIAUCSL   loaded   shape=(767, 9)
  Track Q_su_pce target S&P 500    loaded   shape=(767, 9)
  Track Q_su_pauli target INDPRO     loaded   shape=(767, 15)
  Track Q_su_pauli target PAYEMS     loaded   shape=(767, 15)
  Track Q_su_pauli target CPIAUCSL   loaded   shape=(767, 18)
  Track Q_su_pauli target S&P 500    loaded   shape=(767, 18)
  Track Q_su_bitstring target IN

  Track Q_heisenberg_single_bit target CPIAUCSL   loaded   shape=(767, 13)
  Track Q_heisenberg_single_bit target S&P 500    loaded   shape=(767, 13)
  Track Q_zz_pauli target INDPRO     loaded   shape=(767, 33)
  Track Q_zz_pauli target PAYEMS     loaded   shape=(767, 33)
  Track Q_zz_pauli target CPIAUCSL   loaded   shape=(767, 39)
  Track Q_zz_pauli target S&P 500    loaded   shape=(767, 39)
  Track Q_zz_pce target INDPRO     loaded   shape=(767, 9)
  Track Q_zz_pce target PAYEMS     loaded   shape=(767, 9)
  Track Q_zz_pce target CPIAUCSL   loaded   shape=(767, 9)
  Track Q_zz_pce target S&P 500    loaded   shape=(767, 9)
  Track Q_zz_single_bit target INDPRO     loaded   shape=(767, 11)
  Track Q_zz_single_bit target PAYEMS     loaded   shape=(767, 11)
  Track Q_zz_single_bit target CPIAUCSL   loaded   shape=(767, 13)
  Track Q_zz_single_bit target S&P 500    loaded   shape=(767, 13)
  Track MCR_su_z target INDPRO     loaded   shape=(756, 10)
  Track MCR_su_z target PAYEMS     loa

Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Track A: resuming, 77240 runs already on disk will be skipped


  Saved: raw/random_walk/A.parquet   new=300  total=300
  Saved: raw/AR6/A.parquet   new=300  total=300


  78240/1211184 runs complete · 2s elapsed · ~35m36s left


  Saved: raw/random_walk/A.parquet   new=300  total=600
  Saved: raw/AR6/A.parquet   new=300  total=600


  Saved: raw/random_walk/A.parquet   new=300  total=900
  Saved: raw/AR6/A.parquet   new=300  total=900


...

  Saved: raw/AR6/MCR_su_z.parquet   new=300  total=7200


  Saved: raw/random_walk/MCR_su_z.parquet   new=300  total=7500
  Saved: raw/AR6/MCR_su_z.parquet   new=300  total=7500


  1204808/1211184 runs complete · 2m50s elapsed · ~10s left


  Saved: raw/random_walk/MCR_su_z.parquet   new=224  total=7724
  Saved: raw/AR6/MCR_su_z.parquet   new=224  total=7724
Loop complete. Total rows: 108136 in 2m50s


Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae  \
model   track           target   horizon                                  
AR6     A               CPIAUCSL 1        0.000006  0.001775   0.001775   
                                 3        0.000042  0.004457   0.004457   
                                 6        0.000125  0.007908   0.007908   
                        INDPRO   1        0.000201  0.006012   0.006012   
                                 3        0.000676  0.012879   0.012879   
...                                            ...       ...        ...   
xgboost Q_zz_single_bit PAYEMS   3        0.000135  0.005356   0.005356   
                                 6        0.000267  0.009705   0.009705   
                        S&P 500  1        0.001528  0.028999   0.028999   
                                 3        0.005941  0.057692   0.057692   
                                 6        0.013501  0.087731   0.087731   

                                              rmse      mase     rmspe  
model   track           target   horizon                                
AR6     A               CPIAUCSL 1        0.001775  0.938839  1.347292  
                                 3        0.004457  0.950166  2.048899  
                                 6        0.007908  1.129897  1.292412  
                        INDPRO   1        0.006012  0.941258  4.458508  
                                 3        0.012879  1.062768  3.371093  
...                                            ...       ...       ...  
xgboost Q_zz_single_bit PAYEMS   3        0.005356  1.948483  2.418332  
                                 6        0.009705  1.800484  4.864151  
                        S&P 500  1        0.028999  0.877603  3.797841  
                                 3        0.057692  0.832648  2.196480  
                                 6        0.087731  0.876204  2.025364  

[1872 rows x 6 columns]

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model   track           target   horizon usrec                                  
AR6     A               CPIAUCSL 1       0      0.000005  0.001586   0.001586   
                                         1      0.000019  0.003242   0.003242   
                                 3       0      0.000027  0.003785   0.003785   
                                         1      0.000161  0.009629   0.009629   
                                 6       0      0.000093  0.006961   0.006961   
...                                                  ...       ...        ...   
xgboost Q_zz_single_bit S&P 500  1       1      0.003793  0.046751   0.046751   
                                 3       0      0.004748  0.052313   0.052313   
                                         1      0.015130  0.099119   0.099119   
                                 6       0      0.009575  0.076644   0.076644   
                                         1      0.043579  0.172686   0.172686   

                                                    rmse      mase     rmspe  
model   track           target   horizon usrec                                
AR6     A               CPIAUCSL 1       0      0.001586  0.860288  1.348996  
                                         1      0.003242  1.546022  1.333686  
                                 3       0      0.003785  0.823602  2.077020  
                                         1      0.009629  1.925051  1.826281  
                                 6       0      0.006961  1.001412  1.211426  
...                                                  ...       ...       ...  
xgboost Q_zz_single_bit S&P 500  1       1      0.046751  1.326651  1.306572  
                                 3       0      0.052313  0.759892  2.327300  
                                         1      0.099119  1.393061  1.188816  
                                 6       0      0.076644  0.773357  2.143247  
                                         1      0.172686  1.664228  1.125310  

[3768 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,adaboost,A,CPIAUCSL,1,0,1972-04-01,0.002407,0.003220
1,adaboost,A,CPIAUCSL,1,1,1972-05-01,0.002401,0.002653
2,adaboost,A,CPIAUCSL,1,2,1972-06-01,0.002395,0.002630
3,adaboost,A,CPIAUCSL,1,3,1972-07-01,0.002389,0.002568
4,adaboost,A,CPIAUCSL,1,4,1972-08-01,0.004762,0.002906


mean      sum
model   track                             
AR6     A                   0.001    5.532
        B                   0.002   15.451
        MCR_su_z            0.001    5.139
        Q_heisenberg_pauli  0.001    4.854
        Q_heisenberg_pce    0.002   14.754
...                           ...      ...
xgboost Q_su_pce            0.032  245.333
        Q_su_single_bit     0.029  220.660
        Q_zz_pauli          0.050  386.071
        Q_zz_pce            0.024  188.579
        Q_zz_single_bit     0.027  208.798

[156 rows x 2 columns]